[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/06_pandas_intro/06_6_Pivot_Crosstab.ipynb)

# 06.6: Pivot Tables, Crosstabs, and Descriptive Tools

GroupBy gives you one number per group. But some questions are inherently two-dimensional: how does survival rate vary across both class and sex simultaneously? The answer is a grid — three rows, two columns, a survival rate in each cell. This notebook introduces crosstabs and pivot tables, tools that produce that grid in a single call, along with correlation and outlier-finding tools for a complete descriptive picture of the data.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

,survived,pclass,name,sex,age,sibsp,parch,fare
0,0,3,Mr. Owen Harris Braund,male,22.0,1,0,7.2500
1,1,1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,female,38.0,1,0,71.2833
2,1,3,Miss. Laina Heikkinen,female,26.0,0,0,7.9250
3,1,1,Mrs. Jacques Heath (Lily May Peel) Futrelle,female,35.0,1,0,53.1000
4,0,3,Mr. William Henry Allen,male,35.0,0,0,8.0500


## The Gap GroupBy Leaves

By now you can answer questions like "what is the average survival rate per class?" with `.groupby("pclass")["survived"].mean()`. That gives you one number per class — a single column of results.

But some questions are inherently two-dimensional. *"How does survival rate vary across both class AND sex simultaneously?"* The answer is a grid: three rows (one per class) and two columns (one per sex), with a survival rate in each cell. That grid is what a **pivot table** and a **crosstab** produce.

The goal of this notebook is to build that 2-D view of the Titanic data, and to introduce a few other descriptive tools for understanding a dataset at a glance.

## 1. `pd.crosstab()` — Counts Across Two Categories

`pd.crosstab()` answers the question: *"how many passengers fall into each combination of two categories?"*

Let's start with the most basic question: how many men and women survived vs did not?

In [2]:
pd.crosstab(df["sex"], df["survived"])

survived,0,1
sex,,
female,81,233
male,464,109


The table tells you: 233 women survived and 81 did not; 109 men survived and 468 did not. But raw counts make it hard to compare because there were more men than women aboard. What you really want is *proportions*.

### From Counts to Proportions

The `normalize` parameter converts counts to fractions. The key question is: fractions *of what?*

- `normalize="index"` — each **row** sums to 1. *"Of all women, what fraction survived?"*
- `normalize="columns"` — each **column** sums to 1. *"Of all survivors, what fraction were women?"*
- `normalize="all"` — the whole table sums to 1. *"What fraction of all passengers were women who survived?"*

In [3]:
# Of all passengers in each sex, what fraction survived?
pd.crosstab(df["sex"], df["survived"], normalize="index").round(2)

survived,0,1
sex,,
female,0.26,0.74
male,0.81,0.19


Now the story is clear: 74% of women survived, 19% of men. That's the number you want when the question is "which group had better odds?"

In [4]:
# Of all survivors, what fraction were women?
# normalize='columns' is useful when the question is about the *composition* of a group
pd.crosstab(df["sex"], df["survived"], normalize="columns").round(2)

survived,0,1
sex,,
female,0.15,0.68
male,0.85,0.32


In [5]:
# Add row and column totals with margins=True
pd.crosstab(
    df["pclass"], df["survived"],
    rownames=["Class"],
    colnames=["Survived"],
    normalize="index",
    margins=True,
).round(2)

Survived,0,1
Class,,
1,0.37,0.63
2,0.53,0.47
3,0.76,0.24
All,0.61,0.39


The margins row (`All`) shows the overall survival rate across all classes. The table immediately shows that first-class survival (63%) was more than twice that of third-class (24%).

## 2. `pd.pivot_table()` — Any Aggregation in a 2-D Grid

`crosstab` is convenient for *counting and proportions*. But what if you want the *average fare* broken down by class and sex? Or the *average age*? For any aggregation beyond counting, use `pd.pivot_table()`.

You can think of `pivot_table` as a `.groupby()` whose output is automatically arranged into a 2-D grid instead of a flat list.

In [6]:
# What was the average survival rate for each class-sex combination?
pd.pivot_table(
    df,
    values  = "survived",
    index   = "pclass",
    columns = "sex",
    aggfunc = "mean",
).round(2)

sex,female,male
pclass,,
1,0.97,0.37
2,0.92,0.16
3,0.50,0.14


This is the most important table in the Titanic dataset. Read across any row: women always outsurvived men. Read down any column: higher class means higher survival for both sexes. But third-class women (49%) still outsurvived first-class men (37%) — being female mattered more than being wealthy.

In [7]:
# What was the average fare by class and sex?
pd.pivot_table(
    df,
    values  = "fare",
    index   = "pclass",
    columns = "sex",
    aggfunc = "mean",
).round(2)

sex,female,male
pclass,,
1,106.13,67.23
2,21.97,19.74
3,16.12,12.70


In [8]:
# Add overall row/column averages with margins=True
pd.pivot_table(
    df,
    values       = "survived",
    index        = "pclass",
    columns      = "sex",
    aggfunc      = "mean",
    margins      = True,
    margins_name = "Overall",
).round(2)

sex,female,male,Overall
pclass,,,
1,0.97,0.37,0.63
2,0.92,0.16,0.47
3,0.50,0.14,0.24
Overall,0.74,0.19,0.39


### Multiple Aggregations at Once

In [9]:
# Both the count and the mean — useful to see sample size alongside rate
pd.pivot_table(
    df,
    values  = "fare",
    index   = "pclass",
    columns = "sex",
    aggfunc = ["mean", "count"],
).round(2)

mean         count     
sex     female   male female male
pclass                           
1       106.13  67.23     94  122
2        21.97  19.74     76  108
3        16.12  12.70    144  343

## 3. Choosing the Right Tool

All three tools summarize grouped data. Here is how to decide which to reach for:

| Tool | Reach for it when… |
|---|---|
| `.groupby().agg()` | You need flexible aggregation, non-grid output, or you'll keep working with the result as a DataFrame |
| `pd.crosstab()` | You want **counts or proportions** of two categorical variables — quick exploratory tables |
| `pd.pivot_table()` | You want **any aggregation** displayed as a 2-D grid of two categorical variables |

The same question can be answered with any of the three:

In [10]:
# Three ways to get average survival rate by pclass and sex

# 1. groupby — flat output, needs .unstack() to make it 2-D
r1 = df.groupby(["pclass","sex"])["survived"].mean().unstack("sex").round(2)

# 2. pivot_table — directly 2-D
r2 = pd.pivot_table(df, values="survived", index="pclass", columns="sex", aggfunc="mean").round(2)

print("groupby + unstack:")
print(r1)
print()
print("pivot_table:")
print(r2)

groupby + unstack:
sex     female  male
pclass              
1         0.97  0.37
2         0.92  0.16
3         0.50  0.14

pivot_table:
sex     female  male
pclass              
1         0.97  0.37
2         0.92  0.16
3         0.50  0.14


## 4. Correlation — Which Variables Move Together?

You now have survival rates broken down by class and sex. But how do the numeric variables — age, fare, sibsp, parch — relate to each other and to survival?

`.corr()` computes the **Pearson correlation** between every pair of numeric columns. Correlation ranges from -1 (perfect negative relationship) to +1 (perfect positive relationship). Zero means no linear relationship.

In [11]:
df[["survived", "pclass", "age", "sibsp", "parch", "fare"]].corr().round(2)

,survived,pclass,age,sibsp,parch,fare
survived,1.00,-0.34,-0.06,-0.04,0.08,0.26
pclass,-0.34,1.00,-0.39,0.09,0.02,-0.55
age,-0.06,-0.39,1.00,-0.30,-0.19,0.11
sibsp,-0.04,0.09,-0.30,1.00,0.41,0.16
parch,0.08,0.02,-0.19,0.41,1.00,0.22
fare,0.26,-0.55,0.11,0.16,0.22,1.00


A few things to notice:

- **`pclass` and `survived`** (-0.34): a negative correlation means higher class number (3rd class) goes with lower survival — consistent with what the pivot table showed.
- **`fare` and `survived`** (+0.26): higher fares go with higher survival, but this is largely explained by class. First-class passengers paid more *and* survived more — the two variables are tangled.
- **`fare` and `pclass`** (-0.55): a strong negative correlation, confirming that first-class passengers (pclass=1) paid dramatically more.

Correlation captures *linear* association, not causation. Paying a higher fare didn't cause survival — it was a proxy for class, and class was a proxy for proximity to lifeboats and priority in boarding them.

## 5. Finding Outliers with `.nlargest()` and `.nsmallest()`

Correlation tells you about patterns across the whole dataset. Sometimes you want to zoom in on the extremes — who paid the most? Who was oldest?

In [12]:
# Who paid the highest fares? Did they survive?
df.nlargest(8, "fare")[["name", "pclass", "fare", "age", "survived"]]

,name,pclass,fare,age,survived
257,Miss. Anna Ward,1,512.3292,35.0,1
676,Mr. Thomas Drake Martinez Cardeza,1,512.3292,36.0,1
733,Mr. Gustave J Lesurer,1,512.3292,35.0,1
27,Mr. Charles Alexander Fortune,1,263.0000,19.0,0
87,Miss. Mabel Helen Fortune,1,263.0000,23.0,1
339,Miss. Alice Elizabeth Fortune,1,263.0000,24.0,1
435,Mr. Mark Fortune,1,263.0000,64.0,0
309,Miss. Emily Borie Ryerson,1,262.3750,18.0,1


In [13]:
# Who were the youngest passengers?
df.nsmallest(8, "age")[["name", "pclass", "age", "survived"]]

,name,pclass,age,survived
799,Master. Assad Alexander Thomas,3,0.42,1
751,Master. Viljo Hamalainen,2,0.67,1
466,Miss. Helene Barbara Baclini,3,0.75,1
641,Miss. Eugenie Baclini,3,0.75,1
77,Master. Alden Gates Caldwell,2,0.83,1
827,Master. George Sibley Richards,2,0.83,1
303,Master. Hudson Trevor Allison,1,0.92,1
163,Master. Eino Viljami Panula,3,1.00,0


The oldest passengers who paid the highest fares were almost all first-class — and most survived. The youngest passengers are a mix of classes, and their survival is more mixed.

## 6. A Complete Descriptive Summary

Now we have all the tools. Here is a concise data profile of the Titanic dataset — the kind of summary you'd write before a deeper analysis.

In [14]:
print("=" * 55)
print("TITANIC — DESCRIPTIVE PROFILE")
print("=" * 55)
print(f"\n{len(df):,} passengers | {df['survived'].mean():.0%} survival rate overall")

print("\nSurvival by sex:")
print(df.groupby("sex")["survived"].mean().apply("{:.0%}".format).to_string())

print("\nSurvival by class:")
print(df.groupby("pclass")["survived"].mean().apply("{:.0%}".format).to_string())

print("\nSurvival rate — class × sex:")
print(pd.pivot_table(df, values="survived", index="pclass",
                     columns="sex", aggfunc="mean").round(2).to_string())

print("\nFare (£):")
print(df["fare"].describe().round(2)[["mean","50%","min","max"]].to_string())

print("\nAge (years):")
print(df["age"].describe().round(1)[["mean","50%","min","max"]].to_string())

TITANIC — DESCRIPTIVE PROFILE

887 passengers | 39% survival rate overall

Survival by sex:
sex
female    74%
male      19%

Survival by class:
pclass
1    63%
2    47%
3    24%

Survival rate — class × sex:
sex     female  male
pclass              
1         0.97  0.37
2         0.92  0.16
3         0.50  0.14

Fare (£):
mean     32.31
50%      14.45
min       0.00
max     512.33

Age (years):
mean    29.5
50%     28.0
min      0.4
max     80.0


## What's next

You have now covered the core pandas toolkit: Series, DataFrame basics, filtering and slicing, data cleaning, GroupBy, and pivot tables. Notebook 06.9 is a set of exercises that ask you to use all of these tools on a new dataset, the Palmer Penguins, so the skills become automatic before you move on.